In [1]:
import requests
from pathlib import Path
import pandas as pd
import pyspark
from pyspark.sql import SparkSession, types
from pyspark.sql import functions as F


In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"pyspark version: {pyspark.__version__}")
print(f"spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 20:30:12 WARN Utils: Your hostname, mballo-pc, resolves to a loopback address: 127.0.1.1; using 192.168.1.94 instead (on interface wlp1s0)
26/03/07 20:30:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 20:30:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


pyspark version: 4.1.1
spark version: 4.1.1


In [3]:
# URL data
URL_PREFIX = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download"

def download_taxi_data(taxi_type: str, year: int) -> None:
    errors = []
    for month in range(1, 13):
        fmonth = f"{month:02d}"
        filename = f"{taxi_type}_tripdata_{year}-{fmonth}.csv.gz"
        url = f"{URL_PREFIX}/{taxi_type}/{filename}"
        local_dir = Path(f"data/raw/{taxi_type}/{year}/{fmonth}")
        local_path = local_dir / filename

        if local_path.exists():
            print(f"skipping {filename} (already exists)")
            continue

        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()

            local_dir.mkdir(parents=True, exist_ok=True)
            with open(local_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=10000):
                    f.write(chunk)
            print(f"{filename}")

        except requests.exceptions.HTTPError as e:
            print(f"{filename} — HTTP error: {e}")
            errors.append((taxi_type, year, month, str(e)))
        except requests.exceptions.ConnectionError:
            print(f"{filename} — connection error, skipping")
            errors.append((taxi_type, year, month, "connection error"))
        except Exception as e:
            print(f"{filename} — unexpected error: {e}")
            errors.append((taxi_type, year, month, str(e)))

    if errors:
        print(f"\n {len(errors)} erreur(s) pour {taxi_type}/{year}:")
        for _, _, m, msg in errors:
            print(f"mois {m:02d}: {msg}")


In [4]:
download_taxi_data("green", 2020)
download_taxi_data("yellow", 2020)

skipping green_tripdata_2020-01.csv.gz (already exists)
skipping green_tripdata_2020-02.csv.gz (already exists)
skipping green_tripdata_2020-03.csv.gz (already exists)
skipping green_tripdata_2020-04.csv.gz (already exists)
skipping green_tripdata_2020-05.csv.gz (already exists)
skipping green_tripdata_2020-06.csv.gz (already exists)
skipping green_tripdata_2020-07.csv.gz (already exists)
skipping green_tripdata_2020-08.csv.gz (already exists)
skipping green_tripdata_2020-09.csv.gz (already exists)
skipping green_tripdata_2020-10.csv.gz (already exists)
skipping green_tripdata_2020-11.csv.gz (already exists)
skipping green_tripdata_2020-12.csv.gz (already exists)
skipping yellow_tripdata_2020-01.csv.gz (already exists)
skipping yellow_tripdata_2020-02.csv.gz (already exists)
skipping yellow_tripdata_2020-03.csv.gz (already exists)
skipping yellow_tripdata_2020-04.csv.gz (already exists)
skipping yellow_tripdata_2020-05.csv.gz (already exists)
skipping yellow_tripdata_2020-06.csv.gz (al

In [5]:
download_taxi_data("green", 2021)
download_taxi_data("yellow", 2021)

skipping green_tripdata_2021-01.csv.gz (already exists)
skipping green_tripdata_2021-02.csv.gz (already exists)
skipping green_tripdata_2021-03.csv.gz (already exists)
skipping green_tripdata_2021-04.csv.gz (already exists)
skipping green_tripdata_2021-05.csv.gz (already exists)
skipping green_tripdata_2021-06.csv.gz (already exists)
skipping green_tripdata_2021-07.csv.gz (already exists)
green_tripdata_2021-08.csv.gz — HTTP error: 404 Client Error: Not Found for url: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2021-08.csv.gz
green_tripdata_2021-09.csv.gz — HTTP error: 404 Client Error: Not Found for url: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2021-09.csv.gz
green_tripdata_2021-10.csv.gz — HTTP error: 404 Client Error: Not Found for url: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2021-10.csv.gz
green_tripdata_2021-11.csv.gz — HTTP error: 404 Client Error: Not

In [6]:
# Lire sans schéma pour voir les vraies colonnes
df_check = spark.read \
    .option("header", "true") \
    .csv('./data/raw/yellow/2020/01/yellow_tripdata_2020-01.csv.gz')

print(df_check.columns)
print(len(df_check.columns))

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge']
18


In [10]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [7]:
# Lire sans schéma pour voir les vraies colonnes
df_check = spark.read \
    .option("header", "true") \
    .csv('./data/raw/green/2020/01/green_tripdata_2020-01.csv.gz')

print(df_check.columns)
print(len(df_check.columns))

['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge']
20


In [8]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("trip_type", types.IntegerType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [12]:
def csv_to_parquet(spark, taxi_types, year, schemas):
    errors = []
    for taxi in taxi_types:
        for month in range(1, 13):
            print(f"processing data for {taxi}/{year}/{month:02d}")

            input_file = f"data/raw/{taxi}/{year}/{month:02d}/{taxi}_tripdata_{year}-{month:02d}.csv.gz"
            output_path = f"data/pq/{taxi}/{year}/{month:02d}/"

            if not Path(input_file).exists():
                print(f"fichier source absent, skipping")
                errors.append((taxi, year, month, "fichier source absent"))
                continue

            if Path(output_path).exists():
                print(f"skipping, already exists")
                continue

            try:
                df = spark.read \
                    .option("header", "true") \
                    .schema(schemas[taxi]) \
                    .csv(input_file)

                df.repartition(2) \
                  .write.mode("overwrite") \
                  .parquet(output_path)

                print(f"saved to {output_path}")

            except Exception as e:
                print(f"erreur lors du traitement: {e}")
                errors.append((taxi, year, month, str(e)))

    if errors:
        print(f"\n {len(errors)} erreur(s):")
        for taxi, year, m, msg in errors:
            print(f"{taxi}/{year}/{m:02d}: {msg}")


In [13]:
schemas = {
    "green": green_schema,
    "yellow": yellow_schema
}
taxi_types=["green", "yellow"]

csv_to_parquet(
    spark=spark,
    taxi_types=taxi_types,
    year=2020,
    schemas=schemas
)

processing data for green/2020/01
skipping, already exists
processing data for green/2020/02
skipping, already exists
processing data for green/2020/03
skipping, already exists
processing data for green/2020/04
skipping, already exists
processing data for green/2020/05
skipping, already exists
processing data for green/2020/06
skipping, already exists
processing data for green/2020/07
skipping, already exists
processing data for green/2020/08
skipping, already exists
processing data for green/2020/09
skipping, already exists
processing data for green/2020/10
skipping, already exists
processing data for green/2020/11
skipping, already exists
processing data for green/2020/12
skipping, already exists
processing data for yellow/2020/01
skipping, already exists
processing data for yellow/2020/02
skipping, already exists
processing data for yellow/2020/03
skipping, already exists
processing data for yellow/2020/04
skipping, already exists
processing data for yellow/2020/05
skipping, already

In [14]:
csv_to_parquet(
    spark=spark,
    taxi_types=taxi_types,
    year=2021,
    schemas=schemas
)

processing data for green/2021/01
skipping, already exists
processing data for green/2021/02
skipping, already exists
processing data for green/2021/03
skipping, already exists
processing data for green/2021/04
skipping, already exists
processing data for green/2021/05
skipping, already exists
processing data for green/2021/06
skipping, already exists
processing data for green/2021/07
skipping, already exists
processing data for green/2021/08
fichier source absent, skipping
processing data for green/2021/09
fichier source absent, skipping
processing data for green/2021/10
fichier source absent, skipping
processing data for green/2021/11
fichier source absent, skipping
processing data for green/2021/12
fichier source absent, skipping
processing data for yellow/2021/01
skipping, already exists
processing data for yellow/2021/02
skipping, already exists
processing data for yellow/2021/03
skipping, already exists
processing data for yellow/2021/04
skipping, already exists
processing data f

In [26]:
df_yellow = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/yellow')

In [25]:
df_green = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/green')

In [27]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [28]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [32]:
yellow_columns = set(df_yellow.columns)
common_columns = [col for col in df_green.columns if col in yellow_columns]
print(common_columns)

['VendorID', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'payment_type', 'congestion_surcharge']


In [34]:
df_green_sel = df_green \
    .select(common_columns) \
    .withColumn('service_type', F.lit('green'))

In [35]:
df_yellow_sel = df_yellow \
    .select(common_columns) \
    .withColumn('service_type', F.lit('yellow'))

In [37]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [41]:
df_trips_data.groupBy('service_type').count().show()

[Stage 12:====================================================>   (14 + 1) / 15]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [42]:
df_trips_data.registerTempTable('trips_data')

In [44]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM trips_data
GROUP BY service_type    
""").show()

[Stage 15:====================================================>   (14 + 1) / 15]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [46]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [79]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

In [78]:
df_result.show(2)

[Stage 75:====================================================>   (14 + 1) / 15]

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         127

In [61]:
# registerTempTable est déprécié depuis Spark 2.0. Utilisation de recommandee: createOrReplaceTempView
df_green.createOrReplaceTempView('green')
df_yellow.createOrReplaceTempView('yellow')

In [70]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

df_green_revenue \
    .repartition(20) \
    .write.mode("overwrite") \
    .parquet('data/report/revenue/green')

df_green_revenue.show(5)

[Stage 39:===========================================>              (3 + 1) / 4]

+-------------------+----+-----------------+--------------+
|               hour|zone|           amount|number_records|
+-------------------+----+-----------------+--------------+
|2020-01-24 09:00:00|  81|            59.49|             2|
|2020-01-16 09:00:00| 213|           229.04|             6|
|2020-01-04 21:00:00|  25|           513.83|            32|
|2020-01-10 19:00:00|  66|           545.68|            27|
|2020-01-30 07:00:00|  75|556.6599999999999|            40|
+-------------------+----+-----------------+--------------+
only showing top 5 rows


In [71]:
df_green_revenue.printSchema()      # voir le schéma
df_green_revenue.count()            # nombre de lignes
df_green_revenue.describe().show()  # statistiques de base

root
 |-- hour: timestamp (nullable = true)
 |-- zone: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- number_records: long (nullable = false)



[Stage 48:===========================================>              (3 + 1) / 4]

+-------+------------------+-----------------+-----------------+
|summary|              zone|           amount|   number_records|
+-------+------------------+-----------------+-----------------+
|  count|            744765|           744765|           744765|
|   mean|125.86824166012097|65.17316979183775|3.094201526656059|
| stddev| 75.88271155872924|91.44606127247312|5.787303080941544|
|    min|                 1|            -88.5|                1|
|    max|               265|2281.749999999999|              141|
+-------+------------------+-----------------+-----------------+



In [72]:
df_green_revenue.count() 

744765

In [74]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

df_yellow_revenue \
    .repartition(20) \
    .write.mode("overwrite") \
    .parquet('data/report/revenue/yellow')

df_yellow_revenue.show(5)

[Stage 69:==================================================>     (10 + 1) / 11]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-03 19:00:00| 142| 6023.090000000007|           403|
|2020-01-02 16:00:00| 236|6808.9800000000105|           439|
|2020-01-26 14:00:00| 239| 6541.650000000012|           437|
|2020-01-09 01:00:00| 100| 653.5600000000001|            37|
|2020-01-06 20:00:00|  13|           2054.06|            86|
+-------------------+----+------------------+--------------+
only showing top 5 rows


In [75]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [85]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

In [86]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')
df_join = spark.read.parquet('data/report/revenue/total')

In [87]:
df_zones = spark.read.parquet('zones/')

In [88]:
df_join.show(2)

+-------------------+----+------------+--------------------+------------------+---------------------+
|               hour|zone|green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|   4|        NULL|                NULL|1004.3000000000002|                   57|
|2020-01-01 00:00:00|  10|        NULL|                NULL|             42.41|                    2|
+-------------------+----+------------+--------------------+------------------+---------------------+
only showing top 2 rows


In [89]:
df_zones.show(2)

+----------+-------+--------------+------------+
|LocationID|Borough|          Zone|service_zone|
+----------+-------+--------------+------------+
|         1|    EWR|Newark Airport|         EWR|
|         2| Queens|   Jamaica Bay|   Boro Zone|
+----------+-------+--------------+------------+
only showing top 2 rows


In [90]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [91]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

In [92]:
df_result.show(5)

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-01-01 00:00:00|   4|              NULL|                NULL|1004.3000000000002|                   57|         4|Manhattan|       Alphabet City| Yellow Zone|
|2020-01-01 00:00:00|  10|              NULL|                NULL|             42.41|                    2|        10|   Queens|        Baisley Park|   Boro Zone|
|2020-01-01 00:00:00|  13|              NULL|                NULL|            1214.8|                   56|        13|Manhattan|   Battery Park City| Yellow Zone|
|2020-01-01 00:00:00| 